<a href="https://colab.research.google.com/github/eva-beresin/Protein-Localization/blob/main/BMED_3201_Protein_Localization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# 1. Feature extraction function for one protein (4 images)

def extract_features_for_sample(sample_id, img_dir, resize_dim=(256, 256)):
    channels = ['blue', 'green', 'red', 'yellow']
    imgs = {}

    for ch in channels:
        # Updated extension from .jpg to .png to match the uploaded dataset
        filepath = os.path.join(img_dir, f"{sample_id}_{ch}.png")
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Return None to handle missing files gracefully during parallel execution
            return None

        # Downsample for fast feature extraction while preserving spatial layout
        imgs[ch] = cv2.resize(img, resize_dim).astype(np.float32) / 255.0

    g = imgs['green'].flatten()
    b = imgs['blue'].flatten()
    r = imgs['red'].flatten()
    y = imgs['yellow'].flatten()

    features = []

    # A. Basic Intensity Statistics for each channel
    for arr in [g, b, r, y]:
        features.extend([
            np.mean(arr),
            np.std(arr),
            np.percentile(arr, 75),
            np.percentile(arr, 90),
            np.percentile(arr, 99)
        ])

    # B. Pearson Correlation between Green (protein) and Reference Channels
    def safe_corr(x, z):
        if np.std(x) < 1e-6 or np.std(z) < 1e-6:
            return 0.0
        return np.corrcoef(x, z)[0, 1]

    features.append(safe_corr(g, b))  # Green-Blue (Nucleus) correlation
    features.append(safe_corr(g, r))  # Green-Red (Microtubules) correlation
    features.append(safe_corr(g, y))  # Green-Yellow (ER) correlation
    features.append(safe_corr(b, r))  # Blue-Red reference correlation

    # C. Thresholded Mask Overlap Ratios (Colocalization)
    g_mask = g > np.percentile(g, 85)
    b_mask = b > np.percentile(b, 80)
    r_mask = r > np.percentile(r, 80)
    y_mask = y > np.percentile(y, 80)

    g_sum = np.sum(g_mask) + 1e-6
    features.append(np.sum(g_mask & b_mask) / g_sum)        # % of bright Green inside Nucleus
    features.append(np.sum(g_mask & r_mask) / g_sum)        # % of bright Green on Microtubules
    features.append(np.sum(g_mask & y_mask) / g_sum)        # % of bright Green on ER
    features.append(np.sum(g_mask & (~b_mask)) / g_sum)     # % of bright Green outside Nucleus

    # D. Mean Green intensity inside vs. outside the Nucleus mask
    features.append(np.mean(g[b_mask]) if np.any(b_mask) else 0.0)
    features.append(np.mean(g[~b_mask]) if np.any(~b_mask) else 0.0)
    features.append(np.mean(g[r_mask]) if np.any(r_mask) else 0.0)

    return np.array(features, dtype=np.float32)

# 2. Build feature matrix and train classifier
if __name__ == "__main__":
    DRIVE_FOLDER = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/"
    IMAGE_FOLDER = os.path.join(DRIVE_FOLDER, "protein_loc_project_data")
    LABELS_CSV = os.path.join(DRIVE_FOLDER, "training_labels.csv")

    if not os.path.exists(IMAGE_FOLDER):
        print(f"ERROR: The image directory was not found at {IMAGE_FOLDER}. Please make sure you have uploaded the 'protein_loc_project_data' folder to Google Drive.")
    else:
        df = pd.read_csv(LABELS_CSV, dtype={'sample_id': str})
        df['sample_id'] = df['sample_id'].str.strip()
        sample_ids = df['sample_id'].tolist()

        print(f"Starting parallel feature extraction for {len(sample_ids)} samples using all available CPU cores...")

        # Use ProcessPoolExecutor to run tasks in parallel
        extract_fn = partial(extract_features_for_sample, img_dir=IMAGE_FOLDER)

        with ProcessPoolExecutor() as executor:
            results = list(executor.map(extract_fn, sample_ids))

        # Check if any image loading failed
        X_all = []
        missing_count = 0
        for sid, res in zip(sample_ids, results):
            if res is None:
                missing_count += 1
            else:
                X_all.append(res)

        if missing_count > 0:
            print(f"Warning: Could not load images for {missing_count} samples. Please check your dataset folder.")

        if len(X_all) > 0:
            X_all = np.vstack(X_all)
            print("Feature extraction complete!")

            # Adjust labels dataframe if we had missing files
            if missing_count > 0:
                valid_mask = [res is not None for res in results]
                df_valid = df[valid_mask].reset_index(drop=True)
            else:
                df_valid = df

            # Split into Training (labeled) and Testing (unlabeled)
            train_mask = df_valid['Mitochondria'].notna()
            X_train = X_all[train_mask]
            X_test = X_all[~train_mask]

            y_mito = df_valid.loc[train_mask, 'Mitochondria'].astype(int).values
            y_nucl = df_valid.loc[train_mask, 'Nucleoli'].astype(int).values

            # Encode as 3 mutually exclusive classes: 0 -> (1,0), 1 -> (0,1), 2 -> (1,1)
            y_class = np.zeros(len(y_mito), dtype=int)
            y_class[(y_mito == 0) & (y_nucl == 1)] = 1
            y_class[(y_mito == 1) & (y_nucl == 1)] = 2

            # Evaluate with 5-Fold Stratified Cross-Validation
            print("Evaluating classifier with 5-Fold Cross Validation...")
            clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(clf, X_train, y_class, cv=cv, scoring='accuracy')
            print(f"5-Fold CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

            # Fit on full training set and predict on test samples
            clf.fit(X_train, y_class)
            test_preds = clf.predict(X_test)

            # Map 3-class predictions back to (Mitochondria, Nucleoli) columns
            pred_mito = np.where(test_preds == 1, 0, 1)
            pred_nucl = np.where(test_preds == 0, 0, 1)

            df_valid.loc[~train_mask, 'Mitochondria'] = pred_mito
            df_valid.loc[~train_mask, 'Nucleoli'] = pred_nucl
            df_valid.to_csv("predicted_protein_labels.csv", index=False)
            print("Saved predictions to predicted_protein_labels.csv")

ERROR: The image directory was not found at /content/drive/MyDrive/BMED3201_Projects/Protein_Localization/protein_loc_project_data. Please make sure you have uploaded the 'protein_loc_project_data' folder to Google Drive.


In [ ]:
import os

# Let's check both the main localization folder and the specific image folder path
protein_path = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/"
image_folder_path = os.path.join(protein_path, "protein_loc_project_data")

print("Checking parent folder contents...")
if os.path.exists(protein_path):
    print("Files in Protein_Localization:", sorted(os.listdir(protein_path)))
else:
    print("Parent folder does not exist!")

print("\nChecking image folder...")
if os.path.exists(image_folder_path):
    files = os.listdir(image_folder_path)
    print(f"SUCCESS: Image folder exists!")
    print(f"Total items found in folder: {len(files)}")
    print("First 10 items:", sorted(files)[:10])
else:
    print(f"ERROR: The directory does not exist at: {image_folder_path}")

Checking parent folder contents...
Parent folder does not exist!

Checking image folder...
ERROR: The directory does not exist at: /content/drive/MyDrive/BMED3201_Projects/Protein_Localization/protein_loc_project_data


In [ ]:
import shutil
import os

# Source on Google Drive
drive_img_folder = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/protein_loc_project_data"
# Destination on local Colab runtime (fast SSD)
local_img_folder = "/content/protein_loc_project_data"

if not os.path.exists(local_img_folder):
    print("Copying images from Google Drive to local Colab runtime... This is much faster for processing!")
    # Use system cp for maximum transfer speed
    !cp -r "$drive_img_folder" "$local_img_folder"
    print("Copying complete!")
else:
    print("Local dataset folder already exists!")

# Verify local count
local_count = len(os.listdir(local_img_folder))
print(f"Total items successfully transferred to local storage: {local_count} / 7208")

Copying images from Google Drive to local Colab runtime... This is much faster for processing!
cp: cannot stat '/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/protein_loc_project_data': No such file or directory
Copying complete!


FileNotFoundError: [Errno 2] No such file or directory: '/content/protein_loc_project_data'

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

def extract_features_for_sample(sample_id, img_dir, resize_dim=(256, 256)):
    channels = ['blue', 'green', 'red', 'yellow']
    imgs = {}

    for ch in channels:
        filepath = os.path.join(img_dir, f"{sample_id}_{ch}.png")
        img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        imgs[ch] = cv2.resize(img, resize_dim).astype(np.float32) / 255.0

    g = imgs['green'].flatten()
    b = imgs['blue'].flatten()
    r = imgs['red'].flatten()
    y = imgs['yellow'].flatten()

    features = []

    for arr in [g, b, r, y]:
        features.extend([
            np.mean(arr),
            np.std(arr),
            np.percentile(arr, 75),
            np.percentile(arr, 90),
            np.percentile(arr, 99)
        ])

    def safe_corr(x, z):
        if np.std(x) < 1e-6 or np.std(z) < 1e-6:
            return 0.0
        return np.corrcoef(x, z)[0, 1]

    features.append(safe_corr(g, b))
    features.append(safe_corr(g, r))
    features.append(safe_corr(g, y))
    features.append(safe_corr(b, r))

    g_mask = g > np.percentile(g, 85)
    b_mask = b > np.percentile(b, 80)
    r_mask = r > np.percentile(r, 80)
    y_mask = y > np.percentile(y, 80)

    g_sum = np.sum(g_mask) + 1e-6
    features.append(np.sum(g_mask & b_mask) / g_sum)
    features.append(np.sum(g_mask & r_mask) / g_sum)
    features.append(np.sum(g_mask & y_mask) / g_sum)
    features.append(np.sum(g_mask & (~b_mask)) / g_sum)

    features.append(np.mean(g[b_mask]) if np.any(b_mask) else 0.0)
    features.append(np.mean(g[~b_mask]) if np.any(~b_mask) else 0.0)
    features.append(np.mean(g[r_mask]) if np.any(r_mask) else 0.0)

    return np.array(features, dtype=np.float32)

if __name__ == "__main__":
    DRIVE_FOLDER = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/"
    IMAGE_FOLDER = "/content/protein_loc_project_data"
    LABELS_CSV = os.path.join(DRIVE_FOLDER, "training_labels.csv")

    if not os.path.exists(IMAGE_FOLDER):
        print(f"ERROR: Local folder {IMAGE_FOLDER} not found.")
    else:
        df = pd.read_csv(LABELS_CSV, dtype={'sample_id': str})
        df['sample_id'] = df['sample_id'].str.strip()
        sample_ids = df['sample_id'].tolist()

        print(f"Extracting features for {len(sample_ids)} samples...")

        extract_fn = partial(extract_features_for_sample, img_dir=IMAGE_FOLDER)
        with ProcessPoolExecutor() as executor:
            results = list(executor.map(extract_fn, sample_ids))

        X_all = []
        missing_count = 0
        for sid, res in zip(sample_ids, results):
            if res is None:
                missing_count += 1
            else:
                X_all.append(res)

        if missing_count > 0:
            print(f"Warning: Could not load images for {missing_count} samples.")

        if len(X_all) > 0:
            X_all = np.vstack(X_all)
            print("Feature extraction complete.")

            if missing_count > 0:
                valid_mask = [res is not None for res in results]
                df_valid = df[valid_mask].reset_index(drop=True)
            else:
                df_valid = df

            train_mask = df_valid['Mitochondria'].notna()
            X_train = X_all[train_mask]
            X_test = X_all[~train_mask]

            y_mito = df_valid.loc[train_mask, 'Mitochondria'].astype(int).values
            y_nucl = df_valid.loc[train_mask, 'Nucleoli'].astype(int).values

            y_class = np.zeros(len(y_mito), dtype=int)
            y_class[(y_mito == 0) & (y_nucl == 1)] = 1
            y_class[(y_mito == 1) & (y_nucl == 1)] = 2

            print("Evaluating classifier performance...")
            clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(clf, X_train, y_class, cv=cv, scoring='accuracy')
            print(f"5-Fold CV Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

            clf.fit(X_train, y_class)
            test_preds = clf.predict(X_test)

            pred_mito = np.where(test_preds == 1, 0, 1)
            pred_nucl = np.where(test_preds == 0, 0, 1)

            df_valid.loc[~train_mask, 'Mitochondria'] = pred_mito
            df_valid.loc[~train_mask, 'Nucleoli'] = pred_nucl

            output_csv_path = os.path.join(DRIVE_FOLDER, "predicted_protein_labels.csv")
            df_valid.to_csv(output_csv_path, index=False)
            print(f"Saved final predictions to: {output_csv_path}")

In [ ]:
import os

drive_img_folder = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/protein_loc_project_data"
local_img_folder = "/content/protein_loc_project_data"

if not os.path.exists(local_img_folder):
    print("Copying images from Google Drive to local Colab runtime... This is much faster for processing!")
    !cp -r "$drive_img_folder" "$local_img_folder"
    print("Copying complete!")
else:
    print("Local dataset folder already exists!")

local_count = len(os.listdir(local_img_folder))
print(f"Total items successfully transferred to local storage: {local_count} / 7208")

In [ ]:
import time

# Quick execution test on 50 samples to verify real-time performance
if 'sample_ids' in locals() and os.path.exists("/content/protein_loc_project_data"):
    test_samples = sample_ids[:50]
    print(f"Benchmarking feature extraction on {len(test_samples)} samples...")

    start_time = time.time()
    extract_fn = partial(extract_features_for_sample, img_dir="/content/protein_loc_project_data")

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(extract_fn, test_samples))

    end_time = time.time()
    duration = end_time - start_time
    print(f"Time taken for {len(test_samples)} samples: {duration:.2f} seconds")
    print(f"Estimated total time for {len(sample_ids)} samples: {(duration / 50) * len(sample_ids) / 60:.2f} minutes")
else:
    print("Please run the setup cells above first to mount Drive, copy the images, and define the functions.")

In [ ]:
import os

mydrive_path = "/content/drive/MyDrive/"
try:
    items = os.listdir(mydrive_path)
    print("Items found in MyDrive:")
    for item in sorted(items):
        full_path = os.path.join(mydrive_path, item)
        is_dir = os.path.isdir(full_path)
        print(f"[{'DIR' if is_dir else 'FILE'}] {item}")
except Exception as e:
    print(f"Error reading MyDrive: {e}")

In [ ]:
import os

project_path = "/content/drive/MyDrive/BMED3201_Projects/"
try:
    items = os.listdir(project_path)
    print("Items found in BMED3201_Projects:")
    for item in sorted(items):
        full_path = os.path.join(project_path, item)
        is_dir = os.path.isdir(full_path)
        print(f"[{'DIR' if is_dir else 'FILE'}] {item}")
except Exception as e:
    print(f"Error reading BMED3201_Projects: {e}")

In [ ]:
import os

protein_path = "/content/drive/MyDrive/BMED3201_Projects/Protein_Localization/"
try:
    items = os.listdir(protein_path)
    print("Items found in Protein_Localization:")
    for item in sorted(items):
        full_path = os.path.join(protein_path, item)
        is_dir = os.path.isdir(full_path)
        print(f"[{'DIR' if is_dir else 'FILE'}] {item}")
except Exception as e:
    print(f"Error reading Protein_Localization: {e}")